### Задание 1
Реализуйте слой `CrossAttentionLayer`. Сначала реализуйте функцию `cross_attention` по формуле из урока. Затем, используя написанную функцию, допишите реализацию класса `CrossAttentionLayer`: добавьте инициализацию слоёв, нужные для расчёта матрицы $Q, K, V$ и примените `cross_attention` во время `forward` pass. 

In [13]:
import torch
from torch import nn
from torch.nn import functional as F
import numpy as np


def cross_attention(q, k, v):
    attn_scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(q.size(-1))
    attn_weights = F.softmax(attn_scores, dim=-1)
    outputs = torch.matmul(attn_weights, v)
    return outputs, attn_weights

class CrossAttentionLayer(nn.Module):
  def __init__(self, d_model, d_k, d_v): # добавьте нужные гиперпараметры 
    super().__init__()
    # ВАШ КОД #
    # инициализация слоёв #
    self.q_proj = nn.Linear(d_model, d_k)
    self.k_proj = nn.Linear(d_model, d_k)
    self.v_proj = nn.Linear(d_model, d_v)

  def forward(self, enc_output, dec_output):
    # ВАШ КОД #
    # Реализуйте прямой проход слоя CrossAttention
    q = self.q_proj(dec_output)
    k = self.k_proj(enc_output)
    v = self.v_proj(enc_output)
    output, attn_weights = cross_attention(q, k, v)
    return output, attn_weights


# Проверка (без батчей)
Q = torch.tensor([[10, 0, 0, 0],
     [0, 10, 0, 0]]).float()
K = torch.tensor([[10, 0, 0, 0],
     [0, 10, 0, 0],
     [0, 0, 10, 0]]).float()
V = torch.tensor([[10, 0, 0, 0],
     [0, 20, 0, 0],
     [0, 0, 30, 0]]).float()
assert cross_attention(Q, K, V)[0].sum() == 30 

### Задание 2
Напишите класс `EncoderDecoderWithAttention`, который будет переиспользовать слой из предыдущего задания. Инициализируйте модель и её блоки и допишите `forward pass`, а именно предсказание следующего токена декодером.

In [18]:
class EncoderDecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, start_token_id, max_len):   # добавьте нужные гиперпараметры
        super().__init__()
        # Закончите инициализацию слоев
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.encoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.cross_attn = CrossAttentionLayer(hidden_dim, hidden_dim, hidden_dim)
        self.lm_head = nn.Linear(hidden_dim * 2, vocab_size)  # concat(dec_output, context)
        self.start_token_id = start_token_id  # не забудьте про bos_token для декодера
        self.max_len = max_len

    def forward(self, src, tgt=None):
        batch_size = src.size(0)
        # 1. Шаг энкодера
        embedded_src = self.embedding(src)  # [B, T_src, E]
        encoder_outputs, (h, c) = self.encoder(embedded_src)    # enc_outputs: [B, T_src, H]

        # 2. Цикл декодера (без teacher forcing)
        input_token = torch.full((batch_size,), self.start_token_id, 
                                 dtype=torch.long, device=src.device)

        logits_history = []
        attn_history = []
        dec_hidden, dec_cell = h, c

        for _ in range(self.max_len):
            embedded_t = self.embedding(input_token).unsqueeze(1)       # [B, 1, E]
            dec_output, (dec_hidden, dec_cell) = self.decoder(embedded_t, (dec_hidden, dec_cell))   # [B, 1, H]

            # CrossAttention
            context, attn = self.cross_attn(encoder_outputs, dec_output)  # [B, 1, H], [B, 1, T_src]
            attn_history.append(attn)

            # Конкатенируем decoder state + context
            concat_vec = torch.concat([dec_output, context], dim=-1)    # [B, 1, 2H]
            step_logits = self.lm_head(concat_vec)                      # [B, 1, vocab]
            logits_history.append(step_logits)

            # Берём токен с максимальной вероятностью
            input_token = step_logits.argmax(dim=-1).squeeze(1)         # [B]

        logits = torch.cat(logits_history, dim=1)                       # [B, max_len, vocab]
        attn_history = torch.cat(attn_history, dim=1)                   # [B, max_len, T_src]
        return logits, attn_history

def test_shapes():
    vocab_size = 50
    embed_dim = 16
    hidden_dim = 32
    start_token_id = 1
    max_len = 5

    model = EncoderDecoderWithAttention(vocab_size, embed_dim, hidden_dim,
                                        start_token_id=start_token_id,
                                        max_len=max_len)

    src = torch.randint(0, vocab_size, (2, 7))   # batch=2, src_len=7
    logits, attn = model(src)

    assert logits.shape == (2, max_len, vocab_size), f"Неправильный размер логитов: {logits.shape}"
    assert attn.shape == (2, max_len, src.size(1)), f"Неправильный размер весов внимания: {attn.shape}"
    print("Размеры совпадают - тест пройден")


def test_greedy_generation():
    vocab_size = 10
    embed_dim = 8
    hidden_dim = 16
    start_token_id = 0
    max_len = 3

    model = EncoderDecoderWithAttention(vocab_size, embed_dim, hidden_dim,
                                        start_token_id=start_token_id,
                                        max_len=max_len)

    src = torch.randint(0, vocab_size, (1, 4))   # batch=1
    logits, attn = model(src)

    preds = logits.argmax(dim=-1)  # [1, max_len]
    print("Предсказанная последовательность:", preds.tolist())
    print("Веса внимания:\n", attn)


# Run tests
test_shapes()
test_greedy_generation()

Размеры совпадают - тест пройден
Предсказанная последовательность: [[2, 7, 8]]
Веса внимания:
 tensor([[[0.2478, 0.2531, 0.2513, 0.2478],
         [0.2483, 0.2518, 0.2508, 0.2490],
         [0.2488, 0.2511, 0.2507, 0.2494]]], grad_fn=<CatBackward0>)


### Teacher Forcing
В последнем задании для декодирования вы написали цикл, который берёт токен, предсказанный на предыдущем шаге, и использует его как вход декодера на новом шаге. Во время инференса ничего лучше придумать нельзя — ведь мы ничего не знаем про будущие токены. А что во время обучения? Есть особенности:

1. Во время обучения мы знаем все токены выхода.
1. Необученная модель пока ещё плохо предсказывает следующие токены.
1. Ошибки имеют свойство накапливаться.

Первый пункт даёт возможность организовать обучение эффективнее — применить Teacher Forcing. При обучении декодера на шаге $t$ в качестве входа подаём истинный предыдущий токен $y_{t-1}$ (после сдвига вправо), а не предсказание модели $\tilde{y}_{t-1}$
 . Получается, что на каждом шаге «учитель» исправляет ошибки предыдущего шага, и ошибки в цикле не накапливаются.

### Задание 3
Реализуйте Teacher Forcing в `EncoderDecoderWithAttention`: замените цикл и используйте `tgt`.

In [27]:
class EncoderDecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, start_token_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.encoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.cross_attn = CrossAttentionLayer(hidden_dim, hidden_dim, hidden_dim)
        self.lm_head = nn.Linear(hidden_dim * 2, vocab_size)
        self.start_token_id = start_token_id

    def forward(self, src, tgt):
        """
        src: [B, T_src]
        tgt: [B, T_tgt]   (целевая последовательность с <eos>-токеном в конце)
        """
        batch_size = src.shape[0]
        # 1. Шаг энкодера
        embedded_src = self.embedding(src)
        encoder_outputs, (h, c) = self.encoder(embedded_src)

        # 2. Вход декодера
        start_tokens = torch.full((batch_size, 1), self.start_token_id,
                                  dtype=torch.long, device=src.device)
        decoder_inputs = torch.cat([start_tokens, tgt[:, :-1]], dim=1)  # shift right
        
        # 3. Шаг декодера (с teacher forcing)
        embedded_trg = self.embedding(decoder_inputs)        
        dec_output, (dec_hidden, dec_cell) = self.decoder(embedded_trg, (h, c))
        context, attn = self.cross_attn(encoder_outputs, dec_output)

        # 4. Линейный слой (LM head)
        logits = self.lm_head(torch.concat([dec_output, context], dim=-1))
        return logits, attn


def test_teacher_forcing():
    vocab_size = 20
    embed_dim = 8
    hidden_dim = 16
    start_token_id = 0

    model = EncoderDecoderWithAttention(vocab_size, embed_dim, hidden_dim, start_token_id)

    src = torch.randint(0, vocab_size, (2, 5))   # batch=2, src_len=5
    tgt = torch.randint(0, vocab_size, (2, 6))   # batch=2, tgt_len=6

    logits, attn = model(src, tgt)

    # Check shapes
    assert logits.shape == (2, 6, vocab_size)
    assert attn.shape == (2, 6, src.size(1))
    print("Размерности логитов и весов внимания совпадают")


    # Check loss computation works
    criterion = nn.CrossEntropyLoss()
    loss = criterion(logits.view(-1, vocab_size), tgt.reshape(-1))
    print("Значение лосса:", loss.item())


# Run test
test_teacher_forcing()


Размерности логитов и весов внимания совпадают
Значение лосса: 3.075251340866089


### Смещение экспозиции (Exposure bias)
У Teacher Forcing есть неприятный сопутствующий эффект, который называется смещение экспозиции. Во время инференса распределение входов в декодер начнёт «ехать» — модель будет делать маленькие ошибки, предсказывая каждый токен. При этом на вход следующего шага будет подаваться смещённый (по сравнению с ground truth) токен. 

Модель, обученная в режиме Teacher Forcing, окажется не готова к такому сдвигу — ведь она привыкла получать на вход правильные токены. Поэтому есть риск, что накопление ошибок приведёт к коллапсу декодера, особенно на длинных последовательностях. 

Простой способ полечить это — применять teacher forcing не всегда, а лишь в доле шагов. В остальных случаях — использовать изначальный подход с циклом. 

Другой способ — добавить шум во время обучения, или обучить «как профессор».

### Professor Forcing
Professor Forcing — улучшение поверх Teacher Forcing, призванное улучшить робастность рекуррентных моделей. Идея техники — максимально сблизить скрытые состояния декодера, полученные во время Teacher Forcing и во время простого пошагового декодирования.  

<img src="professor_forcing.png" alt="альтернативный текст" width="700" height="400">

[В оригинальной статье](https://arxiv.org/pdf/1610.09038) для сближения скрытых состояний использовался дискриминатор, который учился различать состояния, полученные разными способами. Но можно использовать и другую разумную меру близости.  

Таким образом, Professor Forcing пытается напрямую бороться со смещением экспозиции на уровне скрытых состояний модели. 